In [ ]:
!pip install -q "transformers>=4.41" shap pandas numpy

import os, numpy as np, pandas as pd, torch, shap, joblib, matplotlib.pyplot as plt
from google.colab import drive

drive.mount("/content/drive")

BASE_DIR    = "/content/drive/MyDrive/longformer_runs/run_paper_v1"
DATA_DIR    = f"{BASE_DIR}/data"
RESULTS_DIR = f"{BASE_DIR}/results_longformer"
SHAP_DIR    = f"{BASE_DIR}/shap_outputs"

os.makedirs(SHAP_DIR, exist_ok=True)

print("DATA_DIR   :", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("SHAP_DIR   :", SHAP_DIR)


Mounted at /content/drive
DATA_DIR   : /content/drive/MyDrive/longformer_runs/run_paper_v1/data
RESULTS_DIR: /content/drive/MyDrive/longformer_runs/run_paper_v1/results_longformer
SHAP_DIR   : /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs


In [ ]:
test_path = os.path.join(DATA_DIR, "test.json")
test_df = pd.read_json(test_path)

def combine_example(code, comment):
    return (code or "") + "\n\n[COMMENT]\n" + (comment or "")

texts = [
    combine_example(code, com)
    for code, com in zip(test_df["new_code_raw"], test_df["new_comment_raw"])
]
labels = test_df["label"].to_numpy()
print("Total test samples:", len(texts))

annot_path = os.path.join(SHAP_DIR, "shap_samples_batched_with_row.csv")
samples_path = os.path.join(SHAP_DIR, "shap_samples_batched.csv")

if os.path.isfile(annot_path):
    shap_samples_df = pd.read_csv(annot_path)
    print("Reusing existing SHAP subset from:", annot_path)
else:
    rng = np.random.default_rng(7)
    TEST_SAMPLE_SIZE = min(30, len(texts))
    idx = rng.choice(len(texts), size=TEST_SAMPLE_SIZE, replace=False)

    sample_texts  = [texts[i] for i in idx]
    sample_labels = labels[idx]

    shap_samples_df = pd.DataFrame({
        "idx_in_test_after_cleaning": idx,
        "text": sample_texts,
        "label": sample_labels
    })
    shap_samples_df.to_csv(samples_path, index=False)
    print("Created new SHAP subset and saved to:", samples_path)

print("SHAP subset size:", len(shap_samples_df))
shap_samples_df.head()


Total test samples: 1066
Reusing existing SHAP subset from: /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs/shap_samples_batched_with_row.csv
SHAP subset size: 30


,idx_in_test_after_cleaning,text,label,shap_row
0,295,public ArrayList<ErrorMsg> getWarnings() {...,1,0
1,298,public int atAdPos(final int pos) {\n ...,0,1
2,361,public List getAnchorHRefs(boolean duplica...,1,2
3,869,\tpublic Index parseAndUpdateIndex(List<JaxbRo...,0,3
4,915,\tprivate boolean isBreakOnOpcode(Integer opco...,0,4


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TextClassificationPipeline

best_ckpt_file = os.path.join(RESULTS_DIR, "BEST_CHECKPOINT.txt")
with open(best_ckpt_file) as f:
    best_ckpt = f.read().strip()

print("Using checkpoint for SHAP:", best_ckpt)

tokenizer = AutoTokenizer.from_pretrained("allenai/longformer-base-4096")
model = AutoModelForSequenceClassification.from_pretrained(best_ckpt)

device = 0 if torch.cuda.is_available() else -1
print("Device:", "GPU" if device == 0 else "CPU")

clf = TextClassificationPipeline(
    model=model,
    tokenizer=tokenizer,
    device=device,
    return_all_scores=True
)

sample_texts = []
sample_labels = []
for i in shap_samples_df["idx_in_test_after_cleaning"]:
    sample_texts.append(texts[i])
    sample_labels.append(labels[i])
sample_labels = np.array(sample_labels)

print("Final SHAP sample_texts:", len(sample_texts))

masker = shap.maskers.Text(tokenizer)

def f_predict(text_list):
    """
    Wrapper for SHAP -> HuggingFace pipeline.
    Handles numpy arrays and label name variations.
    """
    if not isinstance(text_list, list):
        text_list = list(text_list)
    text_list = [str(t) for t in text_list]

    outs = clf(
        text_list,
        batch_size=4,
        truncation=True,
        max_length=1024
    )

    out_mat = np.zeros((len(outs), 2), dtype=float)
    for i, row in enumerate(outs):
        for d in row:
            lab = d["label"]
            if lab.startswith("LABEL_"):
                idx = int(lab.replace("LABEL_", ""))
            else:
                idx = int(lab)
            out_mat[i, idx] = d["score"]
    return out_mat


Using checkpoint for SHAP: /content/drive/MyDrive/longformer_runs/run_paper_v1/results_longformer/checkpoint-1575


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Device: GPU
Final SHAP sample_texts: 30


/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [ ]:
from collections import defaultdict
import re

explainer = shap.Explainer(f_predict, masker)

print("Computing SHAP values on", len(sample_texts), "samples...")
shap_values = explainer(sample_texts, max_evals=500)

print("Type of shap_values:", type(shap_values))
print("len(shap_values):", len(shap_values))


pkl_path = os.path.join(SHAP_DIR, "shap_values_batched.pkl")
joblib.dump((shap_values, sample_texts), pkl_path)
print("Saved SHAP objects to:", pkl_path)

shap_samples_df = shap_samples_df.copy()
shap_samples_df["shap_row"] = np.arange(len(sample_texts))
annot_path = os.path.join(SHAP_DIR, "shap_samples_batched_with_row.csv")
shap_samples_df.to_csv(annot_path, index=False)
print("Saved annotated samples to:", annot_path)



def clean_token(t):
    if t is None:
        return ""
    s = str(t)
    s = s.replace("Ġ", " ").replace("▁", " ").replace("##", "")
    s = re.sub(r"\s+", " ", s).strip()
    if s in {"", "[CLS]", "[SEP]", "<s>", "</s>", "<pad>"}:
        return ""
    return s

token_scores = defaultdict(list)

for e in shap_values:  
    vals = e.values
    toks = e.data


    if vals.ndim == 2:
        vals_class1 = vals[:, 1]
    else:
        vals_class1 = vals

    for tok, v in zip(toks, vals_class1):
        ct = clean_token(tok)
        if ct:
            token_scores[ct].append(abs(float(v)))

if token_scores:
    global_df = pd.DataFrame({
        "token": list(token_scores.keys()),
        "mean_abs_shap": [np.mean(v) for v in token_scores.values()],
        "count": [len(v) for v in token_scores.values()]
    }).sort_values("mean_abs_shap", ascending=False).head(25)

    plt.figure(figsize=(8, max(4, 0.3*len(global_df))))
    y = np.arange(len(global_df))[::-1]
    labels = [f"{t} ({c})" for t, c in zip(global_df["token"], global_df["count"])]
    plt.barh(y, global_df["mean_abs_shap"].values)
    plt.yticks(y, labels, fontsize=9)
    plt.xlabel("Mean |SHAP|")
    plt.title("Global Top Tokens (Class 1)", fontsize=14)
    plt.tight_layout()

    global_fig_path = os.path.join(SHAP_DIR, "global_top_tokens_class1.png")
    plt.savefig(global_fig_path, dpi=200, bbox_inches="tight")
    plt.close()
    print("Saved global bar plot to:", global_fig_path)
else:
    print("No tokens aggregated for global plot.")



max_local = min(10, len(shap_values))

for i in range(max_local):
    
    e = shap_values[i]
    if e.values.ndim == 2:
        
        e_class1 = e[:, 1]
    else:
        e_class1 = e  # fallback

    fp = shap.plots.force(e_class1, matplotlib=False)
    html_path = os.path.join(SHAP_DIR, f"force_plot_batched_{i}.html")
    shap.save_html(html_path, fp)
    print("Saved force plot:", html_path)


Initializing global attention on CLS token...
Input ids are automatically padded to be a multiple of `config.attention_window`: 512


Computing SHAP values on 30 samples...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer:   3%|▎         | 1/30 [00:00<?, ?it/s]

...

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 100%|██████████| 30/30 [17:05<00:00, 41.94s/it]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 31it [17:34, 35.15s/it]


Type of shap_values: <class 'shap._explanation.Explanation'>
len(shap_values): 30
Saved SHAP objects to: /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs/shap_values_batched.pkl
Saved annotated samples to: /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs/shap_samples_batched_with_row.csv
Saved global bar plot to: /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs/global_top_tokens_class1.png
Saved force plot: /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs/force_plot_batched_0.html
Saved force plot: /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs/force_plot_batched_1.html
Saved force plot: /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs/force_plot_batched_2.html
Saved force plot: /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs/force_plot_batched_3.html
Saved force plot: /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs/force_plot_batched_4.html
Saved force pl